# Fire Season Timing - Mediterranean Basin

In [1]:
'''
Computes fire season timing metrics (onset, peak, end, season length) for all WWF RESOLVE ecoregions
intersecting the study region for years 2003-2025.

Data sources:
- MODIS Terra active fire: MODIS/061/MOD14A1
- MODIS Aqua active fire:  MODIS/061/MYD14A1
- Ecoregions:              RESOLVE/ECOREGIONS/2017

Region definition:
- Mediterranean Basin bounding box: lon -10 to 42, lat 28 to 48
- Ecoregion number can be changed by parameters

Output (all paths derived from RUN_LABEL, RUN_VERSION, and BASE_OUT_DIR):
- <BASE_OUT_DIR>/outputs/<RUN_LABEL>_<RUN_VERSION>/<ECO_ID>_<ECO_NAME>.csv
- <BASE_OUT_DIR>/outputs/<RUN_LABEL>_<RUN_VERSION>/daily_counts/
- <BASE_OUT_DIR>/outputs/<RUN_LABEL>_<RUN_VERSION>/_all_metrics.csv
- <BASE_OUT_DIR>/outputs/<RUN_LABEL>_<RUN_VERSION>/_all_daily_counts.csv
- <BASE_OUT_DIR>/outputs/<RUN_LABEL>_<RUN_VERSION>/master_<RUN_LABEL>_<RUN_VERSION>.csv
- <BASE_OUT_DIR>/outputs/<RUN_LABEL>_<RUN_VERSION>/README.txt
- <BASE_OUT_DIR>/outputs/<RUN_LABEL>_<RUN_VERSION>/eco_geometries.json
'''

import ee
import pandas as pd
import matplotlib.pyplot as plt
import os
import time
import datetime
import calendar
import json
from tqdm import tqdm

In [2]:
# Authenticate and initialize ----------------------------------------------------------------------
ee.Authenticate()
ee.Initialize(project='fire-seasons')

In [4]:
# RUN CONFIGURATION --------------------------------------------------------------------------------
# Set these before running anything else. All output paths are derived from these values.

RUN_LABEL   = 'med_basin'  # short name for this run
RUN_VERSION = 'v1'         # increment this for each new run
RUN_NOTES   = """
First test of the new Med Basin script.
"""

BASE_OUT_DIR = r'C:\Users\ibekar\Documents\GitProjects\TGPF'

# Derived paths — do not edit below this line
_run_stamp = datetime.date.today().strftime('%Y-%m-%d')
_run_name  = f'{RUN_LABEL}_{RUN_VERSION}'
output_dir = os.path.join(BASE_OUT_DIR, 'outputs', _run_name)
daily_dir  = os.path.join(output_dir, 'daily_counts')

os.makedirs(output_dir, exist_ok=True)

print(f'Run name  : {_run_name}')
print(f'Output dir: {output_dir}')

Run name  : med_basin_v1
Output dir: C:\Users\ibekar\Documents\GitProjects\TGPF\outputs\med_basin_v1


## Set up

In [5]:
# LOAD MODIS COLLECTIONS ---------------------------------------------------------------------------
# Terra and Aqua are loaded once here at module level.
# Per-year and per-day filtering is handled inside get_daily_counts().

terra = ee.ImageCollection("MODIS/061/MOD14A1").select('FireMask')
aqua  = ee.ImageCollection("MODIS/061/MYD14A1").select('FireMask')

print('Terra image count:', terra.size().getInfo())
print('Aqua image count:', aqua.size().getInfo())
print('Terra and Aqua collections loaded.')

Terra image count: 9447
Aqua image count: 8627
Terra and Aqua collections loaded.


In [6]:
# LOAD MEDITERRANEAN BASIN ECOREGIONS --------------------------------------------------------------
med_bbox = ee.Geometry.BBox(-10, 28, 42, 48)

ecoregions_med = ee.FeatureCollection("RESOLVE/ECOREGIONS/2017") \
                   .filterBounds(med_bbox)

n_eco    = ecoregions_med.size().getInfo()
eco_list = ecoregions_med.select(['ECO_ID', 'ECO_NAME', 'BIOME_NUM', 'BIOME_NAME']).getInfo()

print(f'Number of ecoregions intersecting Mediterranean bounding box: {n_eco}')
print()
for f in eco_list['features']:
    p = f['properties']
    print(p['ECO_ID'], '|', p['ECO_NAME'], '|', p['BIOME_NAME'])

Number of ecoregions intersecting Mediterranean bounding box: 59

701 | Mediterranean conifer and mixed forests | Temperate Conifer Forests
822 | East Sahara Desert | Deserts & Xeric Shrublands
833 | North Saharan Xeric Steppe and Woodland | Deserts & Xeric Shrublands
836 | Red Sea coastal desert | Deserts & Xeric Shrublands
845 | West Sahara desert | Deserts & Xeric Shrublands
745 | Saharan halophytics | Flooded Grasslands & Savannas
744 | Nile Delta flooded savanna | Flooded Grasslands & Savannas
758 | Mediterranean High Atlas juniper steppe | Montane Grasslands & Shrublands
648 | Cantabrian mixed forests | Temperate Broadleaf & Mixed Forests
676 | Pyrenees conifer and mixed forests | Temperate Broadleaf & Mixed Forests
788 | Corsican montane broadleaf and mixed forests | Mediterranean Forests, Woodlands & Scrub
789 | Crete Mediterranean forests | Mediterranean Forests, Woodlands & Scrub
792 | Iberian conifer forests | Mediterranean Forests, Woodlands & Scrub
793 | Iberian sclerophyl

In [7]:
# BUILD ECO RECORDS --------------------------------------------------------------------------------
eco_records = []
for f in eco_list['features']:
    p = f['properties']
    eco_records.append({
        'eco_id'    : p['ECO_ID'],
        'eco_name'  : p['ECO_NAME'],
        'biome_num' : p['BIOME_NUM'],
        'biome_name': p['BIOME_NAME'],
        'geometry'  : ee.Geometry(f['geometry'])
    })

print(f'Built {len(eco_records)} ecoregion records.')

Built 59 ecoregion records.


## Parameters

In [14]:
# PARAMETERS ---------------------------------------------------------------------------------------

FIRE_MASK_MIN   = 8     # FireMask threshold: >= 8 = nominal + high confidence only
ONSET_THRESHOLD = 0.05  # Cumulative fraction threshold for fire season onset (5%)
END_THRESHOLD   = 0.95  # Cumulative fraction threshold for fire season end (95%)
MIN_DETECTIONS  = 20    # Minimum annual fire detections required to compute metrics
YEARS           = list(range(2003, 2026))  # Full study period: 2003–2025

# SUBSETTING (set to None to disable) --------------------------------------------------------------
# Quick test: run only the first N ecoregions (None = all)
TEST_N   = 20

# Target specific ecoregions by ECO_ID (None = all)
# Example: TEST_IDS = [614, 615, 616]
TEST_IDS = None

# APPLY SUBSETTING ---------------------------------------------------------------------------------
eco_run = eco_records

if TEST_IDS is not None:
    eco_run = [e for e in eco_run if e['eco_id'] in TEST_IDS]
    print(f'Subsetting to {len(eco_run)} ecoregions by ID: {TEST_IDS}')

if TEST_N is not None:
    eco_run = eco_run[:TEST_N]
    print(f'Subsetting to first {TEST_N} ecoregions.')

print(f'Running pipeline on {len(eco_run)} / {len(eco_records)} ecoregions.')

Subsetting to first 20 ecoregions.
Running pipeline on 20 / 59 ecoregions.


In [15]:
# SAVE GEOMETRIES TO DISK --------------------------------------------------------------------------
# Saves ecoregion geometries as GeoJSON for reuse in visualization notebooks
# without needing a GEE connection. Skipped if file already exists.
# Uses eco_run — respects subsetting if active, full list if not.

os.makedirs(output_dir, exist_ok=True)
geo_path = os.path.join(output_dir, 'eco_geometries.json')

if os.path.exists(geo_path):
    print(f'Geometries already saved — skipping. ({geo_path})')
else:
    geo_records_export = []
    for rec in eco_run:
        geo_records_export.append({
            'eco_id'  : rec['eco_id'],
            'eco_name': rec['eco_name'],
            'geometry': rec['geometry'].getInfo()
        })

    with open(geo_path, 'w') as f:
        json.dump(geo_records_export, f)

    print(f'Saved {len(geo_records_export)} geometries → {geo_path}')

Saved 20 geometries → C:\Users\ibekar\Documents\GitProjects\TGPF\outputs\med_basin_v1\eco_geometries.json


In [16]:
# WRITE README -------------------------------------------------------------------------------------
_readme_path = os.path.join(output_dir, 'README.txt')
with open(_readme_path, 'w') as _f:
    _f.write(f'Run name    : {_run_name}\n')
    _f.write(f'Date        : {_run_stamp}\n')
    _f.write(f'Years       : {YEARS[0]}–{YEARS[-1]}\n')
    _f.write(f'TEST_N      : {TEST_N}\n')
    _f.write(f'TEST_IDS    : {TEST_IDS}\n')
    _f.write(f'Ecoregions  : {len(eco_run)} / {len(eco_records)}\n')
    _f.write(f'\nNotes:\n{RUN_NOTES.strip()}\n')
print(f'README written → {_readme_path}')

README written → C:\Users\ibekar\Documents\GitProjects\TGPF\outputs\med_basin_v1\README.txt


## Helper Functions

In [19]:
# FUNCTION: get_daily_counts -----------------------------------------------------------------------


def get_daily_counts(eco_geometry, year):
    """
    Compute daily MODIS active fire detection counts for a given
    ecoregion geometry and calendar year.

    Combines Terra (MOD14A1) and Aqua (MYD14A1) by taking the pixel-wise
    maximum across sensors for each day, deduplicating detections that
    appear in both sensors on the same day.

    All 365 daily counts are retrieved in a SINGLE reduceRegion call
    by stacking all daily images into one multi-band image using toBands().
    This avoids the 'Too many concurrent aggregations' error that occurs
    when reduceRegion is called inside a mapped function.

    Parameters
    ----------
    eco_geometry : ee.Geometry
        The geometry of the ecoregion to compute counts for.
    year : int
        The calendar year to process (e.g. 2008).

    Returns
    -------
    pd.DataFrame
        DataFrame with columns:
          - doy           : int, day of year (1-indexed)
          - n_detections  : int, number of fire pixels detected
        One row per day of the year (365 or 366 rows).
    """

    start  = ee.Date.fromYMD(year, 1, 1)
    end    = ee.Date.fromYMD(year + 1, 1, 1)
    n_days = 366 if calendar.isleap(year) else 365

    # Pre-filter both collections to this year
    terra_year = terra.filterDate(start, end)
    aqua_year  = aqua.filterDate(start, end)

    # Fallback empty image for days where a sensor returns no image
    empty = ee.Image.constant(0).rename('FireMask').toUint8()

    # Server-side list of day offsets: [0, 1, 2, ... n_days-1]
    day_seq = ee.List.sequence(0, n_days - 1)

    def make_daily_image(d):
        """
        For a single day offset d, build a deduplicated binary fire image.
        Returns a single-band image named by its DOY (e.g. 'day_001').
        No reduceRegion here — reduction happens once outside this function.
        """
        d        = ee.Number(d)
        date     = start.advance(d, 'day')
        date_end = date.advance(1, 'day')

        terra_day = terra_year.filterDate(date, date_end)
        aqua_day  = aqua_year.filterDate(date, date_end)

        # Use empty fallback if sensor has no image for this day
        t = ee.Image(ee.Algorithms.If(
            terra_day.size().gt(0),
            terra_day.select('FireMask').max(),
            empty
        ))
        a = ee.Image(ee.Algorithms.If(
            aqua_day.size().gt(0),
            aqua_day.select('FireMask').max(),
            empty
        ))

        # Pixel-wise max across sensors = deduplication
        combined    = t.max(a)
        fire_binary = combined.gte(FIRE_MASK_MIN).unmask(0)

        # Name this band by its DOY so we can identify it after toBands()
        band_name = ee.String('day_').cat(
            d.add(1).toInt().format('%03d')
        )

        return fire_binary.rename(band_name)

    # Build a collection of 365 single-band images
    daily_collection = ee.ImageCollection(day_seq.map(make_daily_image))

    # Stack all 365 bands into ONE multi-band image
    stacked = daily_collection.toBands()

    # ONE single reduceRegion call on the entire stacked image
    counts_dict = stacked.reduceRegion(
        reducer   = ee.Reducer.sum(),
        geometry  = eco_geometry,
        scale     = 1000,
        maxPixels = 1e9
    ).getInfo()

    # Parse back into a tidy DataFrame
    rows = []
    for band_name, count in sorted(counts_dict.items()):
        doy = int(band_name[-3:])
        rows.append({
            'doy'         : doy,
            'n_detections': int(count) if count is not None else 0
        })

    return pd.DataFrame(rows)

In [20]:
# FUNCTION: compute_timing_metrics -----------------------------------------------------------------

def compute_timing_metrics(df, year):
    """
    Onset and end: 5%/95% cumulative detection thresholds.
    Peak: fire activity centroid (detection-weighted mean DOY).
    Added fields: onset_month, peak_month, log_n_detections,
                  peak_outside_window.
    Returns None if total detections < MIN_DETECTIONS.
    """

    total = df['n_detections'].sum()

    if total < MIN_DETECTIONS:
        print(f'  {year}: insufficient detections ({total}), skipping.')
        return None

    df = df.copy().sort_values('doy')
    cumulative = df['n_detections'].cumsum()
    cum_frac   = cumulative / total

    onset_rows = df[cum_frac >= ONSET_THRESHOLD]
    end_rows   = df[cum_frac >= END_THRESHOLD]

    if onset_rows.empty or end_rows.empty:
        print(f'  {year}: could not compute onset or end, skipping.')
        return None

    onset_doy = int(onset_rows.iloc[0]['doy'])
    end_doy   = int(end_rows.iloc[0]['doy'])

    weights  = df['n_detections']
    peak_doy = int(round((df['doy'] * weights).sum() / weights.sum()))

    peak_outside_window = not (onset_doy <= peak_doy <= end_doy)
    if peak_outside_window:
        print(f'  {year}: WARNING — peak ({peak_doy}) outside onset-end window '
              f'({onset_doy}-{end_doy}), flagging.')

    season_length = end_doy - onset_doy + 1

    onset_month = (datetime.date(year, 1, 1) + datetime.timedelta(days=onset_doy - 1)).month
    peak_month  = (datetime.date(year, 1, 1) + datetime.timedelta(days=peak_doy  - 1)).month

    return {
        'year'               : year,
        'onset_doy'          : onset_doy,
        'peak_doy'           : peak_doy,
        'end_doy'            : end_doy,
        'season_length'      : season_length,
        'n_detections'       : int(total),
        'onset_month'        : onset_month,
        'peak_month'         : peak_month,
        'peak_outside_window': int(peak_outside_window)
    }

## Main Pipeline

In [21]:
# FULL PIPELINE LOOP - ALL ECOREGIONS x ALL YEARS --------------------------------------------------

os.makedirs(output_dir, exist_ok=True)
os.makedirs(daily_dir,  exist_ok=True)

all_metrics  = []
failed_years = []

for eco in tqdm(eco_run, desc='Ecoregions'):
    eco_id    = eco['eco_id']
    eco_name  = eco['eco_name']
    biome_num = eco['biome_num']
    biome_name= eco['biome_name']
    geometry  = eco['geometry']

    safe_name  = eco_name.replace(' ', '_').replace('/', '_')
    eco_path   = os.path.join(output_dir, f'{eco_id}_{safe_name}.csv')
    daily_path = os.path.join(daily_dir,  f'{eco_id}_{safe_name}_daily.csv')

    # ------------------------------------------------------------------
    # CHECKPOINT — daily file is the single signal that this ecoregion
    # was fully processed. eco_path may be absent if no valid years exist.
    # ------------------------------------------------------------------
    if os.path.exists(daily_path):
        if os.path.exists(eco_path):
            existing = pd.read_csv(eco_path)
            all_metrics.extend(existing.to_dict('records'))
            print(f'  Skipping {eco_name} — already done')
        else:
            print(f'  {eco_name} — daily counts on disk but no metrics file, recomputing locally.')
            saved_daily = pd.read_csv(daily_path)
            eco_metrics = []

            for year in YEARS:
                df_year = saved_daily[saved_daily['year'] == year][['doy', 'n_detections']]
                if df_year.empty:
                    continue
                metrics = compute_timing_metrics(df_year, year)
                if metrics is not None:
                    metrics['eco_id']    = eco_id
                    metrics['eco_name']  = eco_name
                    metrics['biome_num'] = biome_num
                    metrics['biome_name']= biome_name
                    eco_metrics.append(metrics)
                    all_metrics.append(metrics)
                else:
                    failed_years.append({
                        'eco_id': eco_id, 'eco_name': eco_name,
                        'year': year, 'reason': 'metrics_none'
                    })

            n_years_valid   = len(eco_metrics)
            pct_years_valid = round(n_years_valid / len(YEARS), 3)
            for m in eco_metrics:
                m['n_years_valid']   = n_years_valid
                m['pct_years_valid'] = pct_years_valid

            if eco_metrics:
                pd.DataFrame(eco_metrics).to_csv(eco_path, index=False)
                print(f'  Recovered {len(eco_metrics)} metric years from daily file')
            else:
                print(f'  No valid fire years for {eco_name} — confirmed from daily file')

        continue

    # ------------------------------------------------------------------
    # GEE FETCH — only reaches here if daily_path does not exist
    # ------------------------------------------------------------------
    print(f'\n=== {eco_name} (ID: {eco_id}) ===')
    eco_metrics   = []
    daily_records = []

    for year in YEARS:
        t0 = time.time()

        try:
            df_year = get_daily_counts(geometry, year)
        except Exception as e:
            failed_years.append({
                'eco_id': eco_id, 'eco_name': eco_name,
                'year': year, 'reason': f'exception: {e}'
            })
            print(f'  {year}: ERROR — {e}')
            continue

        df_year['year']     = year
        df_year['eco_id']   = eco_id
        df_year['eco_name'] = eco_name
        daily_records.extend(df_year.to_dict('records'))

        metrics = compute_timing_metrics(df_year, year)

        if metrics is not None:
            metrics['eco_id']    = eco_id
            metrics['eco_name']  = eco_name
            metrics['biome_num'] = biome_num
            metrics['biome_name']= biome_name
            eco_metrics.append(metrics)
            all_metrics.append(metrics)
        else:
            failed_years.append({
                'eco_id': eco_id, 'eco_name': eco_name,
                'year': year, 'reason': 'metrics_none'
            })

        t1 = time.time()
        print(f'  {year}: done in {t1 - t0:.1f}s')

    # Save daily counts (always, even if all metric years failed)
    if daily_records:
        daily_df = pd.DataFrame(daily_records)[[
            'eco_id', 'eco_name', 'year', 'doy', 'n_detections'
        ]]
        daily_df.to_csv(daily_path, index=False)
        print(f'  Saved {len(daily_df)} daily rows → {os.path.abspath(daily_path)}')

    # Quality flags across all valid years for this ecoregion
    n_years_valid   = len(eco_metrics)
    pct_years_valid = round(n_years_valid / len(YEARS), 3)
    for m in eco_metrics:
        m['n_years_valid']   = n_years_valid
        m['pct_years_valid'] = pct_years_valid

    # Save metrics CSV
    if eco_metrics:
        eco_df = pd.DataFrame(eco_metrics)
        eco_df.to_csv(eco_path, index=False)
        print(f'  Saved {len(eco_metrics)} metric years → {os.path.abspath(eco_path)}')
    else:
        print(f'  No valid metric years for {eco_name}.')

    # Update failed log after each ecoregion
    if failed_years:
        pd.DataFrame(failed_years).to_csv(
            os.path.join(output_dir, '_failed.csv'), index=False
        )

print('\nAll ecoregions complete.')

Ecoregions:   0%|          | 0/20 [00:00<?, ?it/s]


=== Mediterranean conifer and mixed forests (ID: 701) ===
  2003: done in 6.6s
  2004: done in 10.8s
  2005: done in 7.2s
  2006: done in 8.1s
  2007: done in 13.7s
  2008: done in 7.1s
  2009: done in 15.0s
  2010: done in 7.2s
  2011: done in 7.3s
  2012: done in 7.2s
  2013: done in 8.4s
  2014: done in 7.8s
  2015: done in 7.9s
  2016: done in 7.3s
  2017: done in 10.0s
  2018: done in 16.9s
  2019: done in 7.5s
  2020: done in 10.5s
  2021: done in 15.2s
  2022: done in 8.2s
  2023: done in 6.7s
  2024: done in 11.3s


Ecoregions:   5%|▌         | 1/20 [03:38<1:09:08, 218.36s/it]

  2025: done in 10.3s
  Saved 8401 daily rows → C:\Users\ibekar\Documents\GitProjects\TGPF\outputs\med_basin_v1\daily_counts\701_Mediterranean_conifer_and_mixed_forests_daily.csv
  Saved 23 metric years → C:\Users\ibekar\Documents\GitProjects\TGPF\outputs\med_basin_v1\701_Mediterranean_conifer_and_mixed_forests.csv

=== East Sahara Desert (ID: 822) ===
  2003: done in 38.4s
  2004: done in 49.9s
  2005: done in 66.9s
  2006: done in 88.6s
  2007: done in 56.0s
  2008: done in 34.1s
  2009: done in 27.1s
  2010: done in 37.1s
  2011: done in 78.2s
  2012: done in 71.2s
  2013: done in 51.5s
  2014: done in 29.2s
  2015: done in 45.3s
  2016: done in 37.8s
  2017: done in 38.1s
  2018: done in 58.4s
  2019: done in 45.9s
  2020: done in 44.3s
  2021: done in 120.3s
  2022: done in 60.4s
  2023: done in 115.1s
  2024: done in 63.1s


Ecoregions:  10%|█         | 2/20 [25:25<4:17:42, 859.05s/it]

...

  2025: done in 9.5s
  Saved 8401 daily rows → C:\Users\ibekar\Documents\GitProjects\TGPF\outputs\med_basin_v1\daily_counts\800_Northwest_Iberian_montane_forests_daily.csv
  Saved 23 metric years → C:\Users\ibekar\Documents\GitProjects\TGPF\outputs\med_basin_v1\800_Northwest_Iberian_montane_forests.csv

=== Pindus Mountains mixed forests (ID: 801) ===
  2003: done in 9.7s
  2004: done in 8.6s
  2005: done in 5.1s
  2006: done in 5.6s
  2007: done in 9.2s
  2008: done in 6.9s
  2009: done in 9.6s
  2010: done in 5.4s
  2011: done in 11.1s
  2012: done in 12.6s
  2013: done in 5.8s
  2014: done in 5.3s
  2015: done in 6.2s
  2016: done in 6.7s
  2017: done in 6.7s
  2018: done in 28.5s
  2019: done in 7.6s
  2020: done in 7.3s
  2021: done in 12.3s
  2022: done in 6.3s
  2023: done in 19.3s
  2024: done in 7.4s


Ecoregions:  95%|█████████▌| 19/20 [2:16:00<05:08, 308.59s/it]

  2025: done in 6.7s
  Saved 8401 daily rows → C:\Users\ibekar\Documents\GitProjects\TGPF\outputs\med_basin_v1\daily_counts\801_Pindus_Mountains_mixed_forests_daily.csv
  Saved 23 metric years → C:\Users\ibekar\Documents\GitProjects\TGPF\outputs\med_basin_v1\801_Pindus_Mountains_mixed_forests.csv

=== South Apennine mixed montane forests (ID: 802) ===
  2003: done in 41.2s
  2004: done in 21.7s
  2005: done in 3.9s
  2006: done in 5.8s
  2007: done in 3.7s
  2008: done in 5.5s
  2009: done in 4.6s
  2010: done in 4.6s
  2011: done in 6.7s
  2012: done in 4.8s
  2013: done in 4.7s
  2014: done in 5.9s
  2015: done in 3.8s
  2016: done in 3.8s
  2017: done in 3.9s
  2018: done in 8.3s
  2019: done in 3.9s
  2020: done in 11.2s
  2021: done in 13.9s
  2022: done in 3.2s
  2023: done in 3.0s
  2024: done in 4.3s


Ecoregions: 100%|██████████| 20/20 [2:19:09<00:00, 417.47s/it]

  2025: done in 16.5s
  Saved 8401 daily rows → C:\Users\ibekar\Documents\GitProjects\TGPF\outputs\med_basin_v1\daily_counts\802_South_Apennine_mixed_montane_forests_daily.csv
  Saved 23 metric years → C:\Users\ibekar\Documents\GitProjects\TGPF\outputs\med_basin_v1\802_South_Apennine_mixed_montane_forests.csv

All ecoregions complete.


## Post-run Assembly

In [24]:
# POST-RUN ASSEMBLY — METRICS AND DAILY COUNTS -----------------------------------------------------

import glob

# Assemble metrics
metric_files = sorted(glob.glob(os.path.join(output_dir, '[!_]*.csv')))
if metric_files:
    metrics_combined = pd.concat(
        [pd.read_csv(f) for f in metric_files], ignore_index=True
    )
    metrics_combined.to_csv(os.path.join(output_dir, '_all_metrics.csv'), index=False)
    print(f'Metrics:      {len(metric_files)} files → {len(metrics_combined)} rows')

# Assemble daily counts
daily_files = sorted(glob.glob(os.path.join(daily_dir, '[!_]*_daily.csv')))
if daily_files:
    daily_combined = pd.concat(
        [pd.read_csv(f) for f in daily_files], ignore_index=True
    )
    daily_combined.to_csv(os.path.join(output_dir, '_all_daily_counts.csv'), index=False)
    print(f'Daily counts: {len(daily_files)} files → {len(daily_combined)} rows')

Metrics:      20 files → 832 rows
Daily counts: 20 files → 168020 rows


In [25]:
# COMBINE ALL RESULTS INTO MASTER CSV --------------------------------------------------------------

master_df = pd.DataFrame(all_metrics)

master_df = master_df[[
    'eco_id', 'eco_name', 'biome_num', 'biome_name',
    'year', 'onset_doy', 'peak_doy', 'end_doy', 'season_length',
    'n_detections', 'onset_month', 'peak_month',
    'peak_outside_window', 'n_years_valid', 'pct_years_valid'
]]

master_path = os.path.join(output_dir, f'master_{_run_name}.csv')
master_df.to_csv(master_path, index=False)

print(f'Master CSV saved: {master_df.shape[0]} ecoregion-year rows.')
print(f'Path: {os.path.abspath(master_path)}')
print()
print(master_df.head(10))

Master CSV saved: 416 ecoregion-year rows.
Path: C:\Users\ibekar\Documents\GitProjects\TGPF\outputs\med_basin_v1\master_med_basin_v1.csv

   eco_id                                 eco_name  biome_num  \
0     701  Mediterranean conifer and mixed forests          5   
1     701  Mediterranean conifer and mixed forests          5   
2     701  Mediterranean conifer and mixed forests          5   
3     701  Mediterranean conifer and mixed forests          5   
4     701  Mediterranean conifer and mixed forests          5   
5     701  Mediterranean conifer and mixed forests          5   
6     701  Mediterranean conifer and mixed forests          5   
7     701  Mediterranean conifer and mixed forests          5   
8     701  Mediterranean conifer and mixed forests          5   
9     701  Mediterranean conifer and mixed forests          5   

                  biome_name  year  onset_doy  peak_doy  end_doy  \
0  Temperate Conifer Forests  2003        190       224      243   
1  Tempera